# EDA

This notebook is the shared workspace for lightweight EDA and follow-up ETL checks.
It currently starts with a Stage 03 exploratory pass and reuses the crawler notebook env-loading pattern so `.env` overrides still apply.


In [11]:
# Resolve the project root no matter whether Jupyter starts in the repo root or the notebook directory.
from pathlib import Path
import sys

candidates = [
    Path.cwd().resolve(),
    Path.cwd().resolve().parent,
    Path.cwd().resolve() / "steam-crawler",
    Path.cwd().resolve().parent / "steam-crawler",
]
ROOT_DIR = next(path for path in candidates if (path / "requirements.txt").exists())
REQUIREMENTS_FILE = ROOT_DIR / "requirements.txt"
ROOT_DIR


PosixPath('/Users/gitaalekhyapaul/Documents/[Local] CS5242/cs5242-project/steam-crawler')

In [12]:
# Install every runtime dependency declared in requirements.txt into the current notebook kernel environment.
print(f"Installing notebook dependencies from {REQUIREMENTS_FILE}")
!{sys.executable} -m pip install -r "{REQUIREMENTS_FILE}"

SRC_DIR = ROOT_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

SRC_DIR


Installing notebook dependencies from /Users/gitaalekhyapaul/Documents/[Local] CS5242/cs5242-project/steam-crawler/requirements.txt


PosixPath('/Users/gitaalekhyapaul/Documents/[Local] CS5242/cs5242-project/steam-crawler/src')

In [14]:
# Load the crawler env with overrides enabled, resolve the configured data directory, and validate the Kaggle credentials.
import importlib
import os

import steam_crawler.config as steam_config

steam_config = importlib.reload(steam_config)

ENV_PATH = steam_config.load_project_env(ROOT_DIR, override=True)
DATA_DIR = steam_config.resolve_data_dir(ROOT_DIR)

stage_path_override = os.getenv("STAGE_03_ARCHIVE_PATH")
stage_candidates = []
if stage_path_override:
    override_path = Path(stage_path_override).expanduser()
    if not override_path.is_absolute():
        override_path = (ROOT_DIR / override_path).resolve()
    stage_candidates.append(override_path)

stage_candidates.extend([
    DATA_DIR / "stage_03_apps_with_metadata.zip",
    DATA_DIR / "stage_03_apps_with_metadata.csv.gz",
    DATA_DIR / "stage_03_apps_with_metadata.csv",
])

STAGE_03_PATH = next((path for path in stage_candidates if path.exists()), stage_candidates[0])

required_env = {
    "KAGGLE_USERNAME": os.getenv("KAGGLE_USERNAME"),
    "KAGGLE_API_TOKEN": os.getenv("KAGGLE_API_TOKEN"),
}
missing_env = [name for name, value in required_env.items() if not value]
if missing_env:
    raise RuntimeError(
        "Missing required Kaggle credentials: "
        + ", ".join(missing_env)
        + ". Add them to the environment or to steam-crawler/.env before running this notebook."
    )

os.environ.setdefault("KAGGLE_KEY", required_env["KAGGLE_API_TOKEN"])

if not STAGE_03_PATH.exists():
    checked = ", ".join(str(path) for path in stage_candidates)
    raise FileNotFoundError(f"Stage 03 archive not found. Checked: {checked}")

print(f"ROOT_DIR: {ROOT_DIR}")
print(f"ENV_PATH: {ENV_PATH}")
print(f"DATA_DIR: {DATA_DIR}")
print(f"STAGE_03_PATH: {STAGE_03_PATH}")
print("Kaggle credentials: ready")


ROOT_DIR: /Users/gitaalekhyapaul/Documents/[Local] CS5242/cs5242-project/steam-crawler
ENV_PATH: /Users/gitaalekhyapaul/Documents/[Local] CS5242/cs5242-project/steam-crawler/.env
DATA_DIR: /Users/gitaalekhyapaul/Documents/[Local] CS5242/cs5242-project/steam-crawler/data
STAGE_03_PATH: /Users/gitaalekhyapaul/Documents/[Local] CS5242/cs5242-project/steam-crawler/data/stage_03_apps_with_metadata.csv.gz
Kaggle credentials: ready


In [15]:
# Streaming helpers keep the notebook usable even when the Stage 03 JSON columns are large.
import csv
import gzip
import json
import zipfile
from collections import Counter
from contextlib import contextmanager
from io import TextIOWrapper

import pandas as pd
from IPython.display import display

CSV_FIELD_SIZE_LIMIT_READY = False

def configure_csv_field_size_limit() -> None:
    global CSV_FIELD_SIZE_LIMIT_READY
    if CSV_FIELD_SIZE_LIMIT_READY:
        return
    limit = sys.maxsize
    while True:
        try:
            csv.field_size_limit(limit)
            CSV_FIELD_SIZE_LIMIT_READY = True
            return
        except OverflowError:
            limit //= 10

@contextmanager
def open_stage_03_text(path: Path):
    if path.suffix == ".gz":
        with gzip.open(path, "rt", encoding="utf-8", newline="") as handle:
            yield handle
        return
    if path.suffix == ".zip":
        with zipfile.ZipFile(path) as archive:
            members = [
                info
                for info in archive.infolist()
                if not info.is_dir() and info.filename.lower().endswith(".csv")
            ]
            if not members:
                raise FileNotFoundError(f"No CSV member found inside {path}")
            member = members[0]
            with archive.open(member, "r") as raw_handle:
                with TextIOWrapper(raw_handle, encoding="utf-8", newline="") as handle:
                    yield handle
        return
    with path.open("rt", encoding="utf-8", newline="") as handle:
        yield handle

def stage_03_fieldnames(path: Path) -> list[str]:
    configure_csv_field_size_limit()
    with open_stage_03_text(path) as handle:
        reader = csv.DictReader(handle)
        return reader.fieldnames or []

def iter_stage_03_rows(path: Path):
    configure_csv_field_size_limit()
    with open_stage_03_text(path) as handle:
        yield from csv.DictReader(handle)

def read_stage_03_df(path: Path) -> pd.DataFrame:
    configure_csv_field_size_limit()
    with open_stage_03_text(path) as handle:
        return pd.read_csv(handle)

def parse_details_data(raw_details_json: str) -> dict:
    if not isinstance(raw_details_json, str) or not raw_details_json:
        return {}
    try:
        payload = json.loads(raw_details_json)
    except json.JSONDecodeError:
        return {}
    if not isinstance(payload, dict) or not payload:
        return {}
    app_payload = next(iter(payload.values()), {})
    if not isinstance(app_payload, dict):
        return {}
    details_data = app_payload.get("data", {})
    return details_data if isinstance(details_data, dict) else {}

def extract_category_ids(details_data: dict):
    categories = details_data.get("categories") or []
    if not isinstance(categories, list):
        return pd.NA
    category_ids = [
        str(category.get("id"))
        for category in categories
        if isinstance(category, dict) and category.get("id") is not None
    ]
    return "|".join(category_ids) if category_ids else pd.NA

def extract_recommendations_total(details_data: dict):
    recommendations = details_data.get("recommendations") or {}
    total = recommendations.get("total") if isinstance(recommendations, dict) else None
    try:
        return int(total)
    except (TypeError, ValueError):
        return pd.NA

def extract_price(details_data: dict):
    if details_data.get("is_free") is True:
        return 0.0
    price_overview = details_data.get("price_overview") or {}
    final_price = price_overview.get("final") if isinstance(price_overview, dict) else None
    try:
        return int(final_price) / 100
    except (TypeError, ValueError):
        return pd.NA

def preview_rows(path: Path, limit: int = 5) -> pd.DataFrame:
    rows = []
    for index, row in enumerate(iter_stage_03_rows(path)):
        compact_row = dict(row)
        for field in ("raw_app_json", "raw_details_json"):
            value = compact_row.get(field, "")
            if isinstance(value, str) and len(value) > 180:
                compact_row[field] = value[:180] + "..."
        rows.append(compact_row)
        if index + 1 >= limit:
            break
    return pd.DataFrame(rows)

def summarize_stage_03(path: Path) -> tuple[int, pd.DataFrame, dict[str, Counter], pd.Series]:
    fieldnames = stage_03_fieldnames(path)
    non_empty_counts = Counter()
    counters = {
        "details_success": Counter(),
        "type": Counter(),
        "eligible_for_sampling": Counter(),
    }
    recommendation_values = []
    total_rows = 0

    for row in iter_stage_03_rows(path):
        total_rows += 1
        for key, value in row.items():
            if value not in (None, ""):
                non_empty_counts[key] += 1
        counters["details_success"][row.get("details_success", "<blank>") or "<blank>"] += 1
        counters["type"][row.get("type", "<blank>") or "<blank>"] += 1
        counters["eligible_for_sampling"][row.get("eligible_for_sampling", "<blank>") or "<blank>"] += 1

        raw_recommendations = row.get("recommendations_total", "")
        if raw_recommendations not in (None, ""):
            try:
                recommendation_values.append(int(raw_recommendations))
            except ValueError:
                pass

    column_summary = pd.DataFrame(
        [
            {
                "column": column,
                "non_empty_rows": non_empty_counts[column],
                "null_or_blank_rows": total_rows - non_empty_counts[column],
            }
            for column in fieldnames
        ]
    )

    if recommendation_values:
        recommendations = pd.Series(recommendation_values, dtype="int64")
        recommendation_profile = pd.Series(
            {
                "non_null_rows": int(recommendations.count()),
                "min": int(recommendations.min()),
                "median": float(recommendations.median()),
                "mean": float(recommendations.mean()),
                "max": int(recommendations.max()),
            },
            name="recommendations_total",
        )
    else:
        recommendation_profile = pd.Series(dtype="float64", name="recommendations_total")

    return total_rows, column_summary, counters, recommendation_profile


In [16]:
row_count, column_summary_df, counters, recommendation_profile = summarize_stage_03(STAGE_03_PATH)
preview_df = preview_rows(STAGE_03_PATH, limit=5)
fieldnames = stage_03_fieldnames(STAGE_03_PATH)

print(f"Stage 03 rows: {row_count:,}")
print(f"Stage 03 columns ({len(fieldnames)}): {fieldnames}")

display(preview_df)
display(column_summary_df)


Stage 03 rows: 162,305
Stage 03 columns (12): ['appid', 'name', 'last_modified', 'price_change_number', 'raw_app_json', 'details_success', 'type', 'category_ids', 'category_descriptions', 'recommendations_total', 'raw_details_json', 'eligible_for_sampling']


,appid,name,last_modified,price_change_number,raw_app_json,details_success,type,category_ids,category_descriptions,recommendations_total,raw_details_json,eligible_for_sampling
0,10,Counter-Strike,1745368572,34672985,"{""appid"":10,""name"":""Counter-Strike"",""last_modi...",True,game,1|49|36|37|66|68|75|69|8|62,Multi-player|PvP|Online PvP|Shared/Split Scree...,167351,"{""10"":{""success"":true,""data"":{""type"":""game"",""n...",True
1,20,Team Fortress Classic,1745368565,34672985,"{""appid"":20,""name"":""Team Fortress Classic"",""la...",True,game,1|49|36|37|68|75|69|8|44|62,Multi-player|PvP|Online PvP|Shared/Split Scree...,6883,"{""20"":{""success"":true,""data"":{""type"":""game"",""n...",True
2,30,Day of Defeat,1745368580,34672985,"{""appid"":30,""name"":""Day of Defeat"",""last_modif...",True,game,1|67|66|68|69|8|62,Multi-player|Camera Comfort|Color Alternatives...,4419,"{""30"":{""success"":true,""data"":{""type"":""game"",""n...",True
3,40,Deathmatch Classic,1745368570,34672985,"{""appid"":40,""name"":""Deathmatch Classic"",""last_...",True,game,1|49|36|37|66|68|75|69|8|44|62,Multi-player|PvP|Online PvP|Shared/Split Scree...,2403,"{""40"":{""success"":true,""data"":{""type"":""game"",""n...",True
4,50,Half-Life: Opposing Force,1745368539,34672985,"{""appid"":50,""name"":""Half-Life: Opposing Force""...",True,game,2|1|68|78|74|79|8|62,Single-player|Multi-player|Custom Volume Contr...,24416,"{""50"":{""success"":true,""data"":{""type"":""game"",""n...",True


,column,non_empty_rows,null_or_blank_rows
0,appid,162305,0
1,name,162287,18
2,last_modified,162305,0
3,price_change_number,162305,0
4,raw_app_json,162305,0
5,details_success,162304,1
6,type,162103,202
7,category_ids,162076,229
8,category_descriptions,162076,229
9,recommendations_total,21382,140923


In [17]:
for label in ["details_success", "type", "eligible_for_sampling"]:
    print(f"\n{label}")
    count_df = pd.DataFrame(
        sorted(counters[label].items(), key=lambda item: (-item[1], item[0])),
        columns=["value", "count"],
    )
    display(count_df)

if recommendation_profile.empty:
    print("recommendations_total is blank for every row")
else:
    display(recommendation_profile.to_frame(name="value"))



details_success


,value,count
0,True,162103
1,False,201
2,<blank>,1



type


,value,count
0,game,162102
1,<blank>,202
2,advertising,1



eligible_for_sampling


,value,count
0,False,152707
1,True,9598


,value
non_null_rows,2.138200e+04
min,1.010000e+02
median,4.070000e+02
mean,5.122290e+03
max,5.029425e+06


In [18]:
stage_03_df = read_stage_03_df(STAGE_03_PATH)
print(f"Loaded Stage 03 dataframe: {stage_03_df.shape[0]:,} rows x {stage_03_df.shape[1]} columns")
display(stage_03_df[["appid", "category_ids", "recommendations_total", "raw_details_json"]].head())


Loaded Stage 03 dataframe: 162,305 rows x 12 columns


/var/folders/fg/p6f6x7z11qb2jtk9d1gdsdlw0000gn/T/ipykernel_21935/1765059389.py:65: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(handle)


,appid,category_ids,recommendations_total,raw_details_json
0,10,1|49|36|37|66|68|75|69|8|62,167351.0,"{""10"":{""success"":true,""data"":{""type"":""game"",""n..."
1,20,1|49|36|37|68|75|69|8|44|62,6883.0,"{""20"":{""success"":true,""data"":{""type"":""game"",""n..."
2,30,1|67|66|68|69|8|62,4419.0,"{""30"":{""success"":true,""data"":{""type"":""game"",""n..."
3,40,1|49|36|37|66|68|75|69|8|44|62,2403.0,"{""40"":{""success"":true,""data"":{""type"":""game"",""n..."
4,50,2|1|68|78|74|79|8|62,24416.0,"{""50"":{""success"":true,""data"":{""type"":""game"",""n..."


In [19]:
# price uses raw_details_json -> data -> price_overview -> final, converted from cents to currency units. Free games are set to 0.0.
details_data_series = stage_03_df["raw_details_json"].apply(parse_details_data)

stage_03_features_df = pd.DataFrame(
    {
        "appid": pd.to_numeric(stage_03_df["appid"], errors="coerce").astype("Int64"),
        "category_ids": stage_03_df["category_ids"].astype("string"),
        "recommendations_total": pd.to_numeric(stage_03_df["recommendations_total"], errors="coerce").astype("Int64"),
        "price": details_data_series.apply(extract_price),
    }
)

missing_category_ids = stage_03_features_df["category_ids"].isna() | stage_03_features_df["category_ids"].str.len().fillna(0).eq(0)
stage_03_features_df.loc[missing_category_ids, "category_ids"] = details_data_series[missing_category_ids].apply(extract_category_ids)

missing_recommendations = stage_03_features_df["recommendations_total"].isna()
stage_03_features_df.loc[missing_recommendations, "recommendations_total"] = details_data_series[missing_recommendations].apply(extract_recommendations_total)

stage_03_features_df["recommendations_total"] = pd.to_numeric(stage_03_features_df["recommendations_total"], errors="coerce").astype("Int64")
stage_03_features_df["price"] = pd.to_numeric(stage_03_features_df["price"], errors="coerce").astype("Float64")

print(f"Extracted dataframe rows: {stage_03_features_df.shape[0]:,}")
display(stage_03_features_df.head())
display(stage_03_features_df.isna().sum().rename("null_count").to_frame())


Extracted dataframe rows: 162,305


,appid,category_ids,recommendations_total,price
0,10,1|49|36|37|66|68|75|69|8|62,167351,<NA>
1,20,1|49|36|37|68|75|69|8|44|62,6883,<NA>
2,30,1|67|66|68|69|8|62,4419,<NA>
3,40,1|49|36|37|66|68|75|69|8|44|62,2403,<NA>
4,50,2|1|68|78|74|79|8|62,24416,<NA>


,null_count
appid,0
category_ids,229
recommendations_total,140923
price,141591


## Stage 4a Patch

This patch uses the sampled Stage 4 CSV as the source of app ids, recommendations totals, and category ids.
It re-fetches only the missing store metadata needed for price and one review page per game to capture the review summary.


In [21]:
import steam_crawler.stage4a as steam_stage4a

STAGE_04A_ENDPOINT_MODE = os.getenv("STAGE_04A_ENDPOINT_MODE") or None
STAGE_04A_FORCE_REFRESH = False

stage_04a_csv_df = steam_stage4a.build_stage_04a(
    ROOT_DIR,
    force_refresh=STAGE_04A_FORCE_REFRESH,
    endpoint_mode=STAGE_04A_ENDPOINT_MODE,
)

display(stage_04a_csv_df.head())
display(stage_04a_csv_df.describe(include="all").transpose())
print(f"stage_04a csv: {ROOT_DIR / 'data' / 'stage_04a_selected_games.csv'}")


Stage 4a patch:   6%|5         | 553/9598 [00:00<?, ?apps/s]

2026-04-18 17:15:43,532 | WARNING | HTTP error | stage=stage_04a_metadata | appid=115110 | attempt=1 | status=429 | retry_after=301.0 | exception=HTTPError: 429 Client Error: Too Many Requests for url: https://store.steampowered.com/api/appdetails?appids=115110&cc=us&l=english&filters=basic%2Cprice_overview | headers={'Server': 'nginx', 'Content-Type': 'application/json; charset=utf-8', 'Content-Encoding': 'gzip', 'Vary': 'Accept-Encoding', 'Strict-Transport-Security': 'max-age=63072000', 'Content-Length': '24', 'Date': 'Sat, 18 Apr 2026 09:15:43 GMT', 'Connection': 'keep-alive'} | body=null


KeyboardInterrupt: 

In [ ]:
stage_04a_df = steam_stage4a.write_stage_04a_parquet(
    ROOT_DIR,
    endpoint_mode=STAGE_04A_ENDPOINT_MODE,
)

display(stage_04a_df.head())
display(stage_04a_df.describe(include="all").transpose())
print(f"stage_04a parquet: {ROOT_DIR / 'data' / 'stage_04a_selected_games.parquet'}")
